# 📖 Notebook 4: Rebalancing Strategies

Your system is growing. One shard is getting too large or too hot. You need to **rebalance** — redistribute data across shards so no single shard is overwhelmed.

Rebalancing is one of the hardest operational challenges with sharding. You need to move data between live databases without downtime or data loss.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why and when rebalancing is needed
- How to detect imbalance across shards
- Online data migration between shards
- Strategies: shard splitting, consistent hashing migration, and directory-based routing

## 🛠️ Setup

Make sure infrastructure is running:

```bash
cd 01-foundations/sharding
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import hashlib
import bisect
import random
import time

SHARD_CONFIGS = {
    0: {"host": "localhost", "port": 55433, "database": "shard_1", "user": "demo", "password": "demo"},
    1: {"host": "localhost", "port": 55434, "database": "shard_2", "user": "demo", "password": "demo"},
    2: {"host": "localhost", "port": 55435, "database": "shard_3", "user": "demo", "password": "demo"},
}

COUNTRIES = ['US', 'UK', 'Germany', 'Japan', 'Brazil', 'India', 'Canada', 'France']

def get_connection(shard_id):
    return psycopg2.connect(**SHARD_CONFIGS[shard_id])

def hash_shard(key, num_shards):
    key_bytes = str(key).encode('utf-8')
    hash_digest = hashlib.md5(key_bytes).hexdigest()
    return int(hash_digest[:8], 16) % num_shards

for sid in SHARD_CONFIGS:
    conn = get_connection(sid)
    cur = conn.cursor()
    cur.execute("SELECT current_database()")
    print(f"✅ Shard {sid} → {cur.fetchone()[0]}")
    cur.close()
    conn.close()

## Step 1: Create an Imbalanced State

First, let's intentionally create an **imbalanced** system. We'll put way more data on Shard 0 to simulate a real-world scenario where one shard has grown larger than the others.

In [ ]:
# Clear all shards
for sid in SHARD_CONFIGS:
    conn = get_connection(sid)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute("DELETE FROM orders")
    cur.execute("DELETE FROM users")
    cur.close()
    conn.close()

# Intentionally imbalanced: put 200 users on shard 0, 50 on shard 1, 50 on shard 2
user_id = 1
shard_assignments = {}  # user_id → shard_id (our directory)

for shard_id, count in [(0, 200), (1, 50), (2, 50)]:
    conn = get_connection(shard_id)
    conn.autocommit = True
    cur = conn.cursor()
    for _ in range(count):
        cur.execute(
            "INSERT INTO users (id, username, email, country) VALUES (%s, %s, %s, %s)",
            (user_id, f"user_{user_id}", f"user{user_id}@example.com", random.choice(COUNTRIES))
        )
        shard_assignments[user_id] = shard_id
        user_id += 1
    cur.close()
    conn.close()

# Also add some orders on shard 0 to simulate write load
products = ['Widget', 'Gadget', 'Doohickey', 'Thingamajig', 'Whatchamacallit']
conn = get_connection(0)
conn.autocommit = True
cur = conn.cursor()
for i in range(1, 501):
    cur.execute(
        "INSERT INTO orders (id, user_id, product, amount) VALUES (%s, %s, %s, %s)",
        (i, random.randint(1, 200), random.choice(products), round(random.uniform(10, 500), 2))
    )
cur.close()
conn.close()

print("Created imbalanced state:")
print("─" * 40)

## Step 2: Detect Imbalance

Before rebalancing, you need to **measure** the imbalance. In production, you'd monitor metrics like row count, storage size, query latency, and CPU usage per shard.

In [ ]:
def get_shard_stats():
    """Collect stats from each shard to detect imbalance."""
    stats = {}
    for sid in SHARD_CONFIGS:
        conn = get_connection(sid)
        cur = conn.cursor()
        
        cur.execute("SELECT COUNT(*) FROM users")
        user_count = cur.fetchone()[0]
        
        cur.execute("SELECT COUNT(*) FROM orders")
        order_count = cur.fetchone()[0]
        
        # Measure query time as a proxy for load
        start = time.time()
        cur.execute("SELECT COUNT(*) FROM users u JOIN orders o ON u.id = o.user_id")
        cur.fetchone()
        query_ms = (time.time() - start) * 1000
        
        stats[sid] = {
            'users': user_count,
            'orders': order_count,
            'query_ms': query_ms
        }
        cur.close()
        conn.close()
    return stats

stats = get_shard_stats()

print("📊 Shard Health Report")
print("═" * 60)
print(f"{'Shard':<8} {'Users':<10} {'Orders':<10} {'Query Time':<12} {'Status'}")
print("─" * 60)

total_users = sum(s['users'] for s in stats.values())
ideal_per_shard = total_users / len(stats)

for sid, s in stats.items():
    ratio = s['users'] / ideal_per_shard if ideal_per_shard > 0 else 0
    if ratio > 1.5:
        status = '🔴 OVERLOADED'
    elif ratio < 0.5:
        status = '🟡 UNDERUSED'
    else:
        status = '🟢 OK'
    print(f"Shard {sid:<3} {s['users']:<10} {s['orders']:<10} {s['query_ms']:>8.1f} ms  {status}")

print()
print(f"Ideal distribution: ~{ideal_per_shard:.0f} users per shard")
print(f"Shard 0 has {stats[0]['users']/ideal_per_shard:.1f}x the ideal amount — time to rebalance!")

## Step 3: Strategy 1 — Online Data Migration

The most straightforward rebalancing approach: move rows from the overloaded shard to underloaded shards. This is called **online migration** because the system stays running during the move.

The process:
1. Read rows from the source shard
2. Insert them into the destination shard
3. Delete from the source shard
4. Update the routing directory

In [ ]:
def migrate_users(source_shard, dest_shard, user_ids, shard_assignments):
    """
    Move users (and their orders) from one shard to another.
    
    In production, you'd do this in batches with error handling.
    This simplified version moves all users at once to show the concept.
    """
    src_conn = get_connection(source_shard)
    dst_conn = get_connection(dest_shard)
    src_conn.autocommit = True
    dst_conn.autocommit = True
    src_cur = src_conn.cursor()
    dst_cur = dst_conn.cursor()
    
    migrated = 0
    for uid in user_ids:
        # 1. Read user from source
        src_cur.execute("SELECT id, username, email, country FROM users WHERE id = %s", (uid,))
        user = src_cur.fetchone()
        if not user:
            continue
        
        # 2. Insert into destination
        dst_cur.execute(
            "INSERT INTO users (id, username, email, country) VALUES (%s, %s, %s, %s)",
            user
        )
        
        # 3. Move any orders too
        src_cur.execute("SELECT id, user_id, product, amount, status FROM orders WHERE user_id = %s", (uid,))
        orders = src_cur.fetchall()
        for order in orders:
            dst_cur.execute(
                "INSERT INTO orders (id, user_id, product, amount, status) VALUES (%s, %s, %s, %s, %s)",
                order
            )
        
        # 4. Delete from source (orders first due to foreign key)
        src_cur.execute("DELETE FROM orders WHERE user_id = %s", (uid,))
        src_cur.execute("DELETE FROM users WHERE id = %s", (uid,))
        
        # 5. Update routing directory
        shard_assignments[uid] = dest_shard
        migrated += 1
    
    src_cur.close()
    dst_cur.close()
    src_conn.close()
    dst_conn.close()
    
    return migrated

# Move 50 users from Shard 0 to Shard 1, and 50 to Shard 2
# Pick users from shard 0
users_on_shard0 = [uid for uid, sid in shard_assignments.items() if sid == 0]
batch_1 = users_on_shard0[:50]   # move to shard 1
batch_2 = users_on_shard0[50:100] # move to shard 2

print("🔄 Migrating users...")
start = time.time()

m1 = migrate_users(0, 1, batch_1, shard_assignments)
print(f"  Moved {m1} users from Shard 0 → Shard 1")

m2 = migrate_users(0, 2, batch_2, shard_assignments)
print(f"  Moved {m2} users from Shard 0 → Shard 2")

elapsed = (time.time() - start) * 1000
print(f"\n⏱️  Migration completed in {elapsed:.0f} ms")

In [ ]:
# Check the new balance
stats_after = get_shard_stats()

print("📊 After Rebalancing")
print("═" * 55)
print(f"{'Shard':<8} {'Before':<12} {'After':<12} {'Change':<12} {'Status'}")
print("─" * 55)

for sid in SHARD_CONFIGS:
    before = stats[sid]['users']
    after = stats_after[sid]['users']
    diff = after - before
    sign = '+' if diff > 0 else ''
    
    ratio = after / ideal_per_shard if ideal_per_shard > 0 else 0
    status = '🟢 Balanced' if 0.7 < ratio < 1.3 else '🔴 Still off'
    
    print(f"Shard {sid:<3} {before:<12} {after:<12} {sign}{diff:<11} {status}")

print()
print("✅ Shards are now much more balanced!")

## Step 4: Strategy 2 — Directory-Based Routing

Instead of computing which shard a key belongs to, you **look it up in a table**. This gives you maximum flexibility — you can move any key to any shard at any time just by updating the directory.

The tradeoff: every request needs an extra lookup, and the directory is a single point of failure.

In [ ]:
class DirectoryRouter:
    """
    Routes keys using a lookup table (directory).
    Maximum flexibility: any key can be on any shard.
    """
    
    def __init__(self, shard_configs):
        self.shard_configs = shard_configs
        self.directory = {}  # key → shard_id
    
    def assign(self, key, shard_id):
        """Assign a key to a specific shard."""
        self.directory[key] = shard_id
    
    def get_shard(self, key):
        """Look up which shard a key belongs to."""
        if key not in self.directory:
            raise KeyError(f"Key {key} not in directory!")
        return self.directory[key]
    
    def move_key(self, key, new_shard):
        """Reassign a key to a different shard (just updates the directory)."""
        old_shard = self.directory.get(key)
        self.directory[key] = new_shard
        return old_shard
    
    def get_stats(self):
        """How many keys per shard."""
        counts = {}
        for shard_id in self.directory.values():
            counts[shard_id] = counts.get(shard_id, 0) + 1
        return counts

# Build directory from our current shard assignments
dir_router = DirectoryRouter(SHARD_CONFIGS)
for uid, sid in shard_assignments.items():
    dir_router.assign(uid, sid)

print("Directory-based routing:")
print("─" * 40)
for sid, count in sorted(dir_router.get_stats().items()):
    print(f"  Shard {sid}: {count} users")

print()

# Demonstrate easy key movement
uid_to_move = 1
old = dir_router.get_shard(uid_to_move)
dir_router.move_key(uid_to_move, 2)
new = dir_router.get_shard(uid_to_move)
print(f"Moved user {uid_to_move}: Shard {old} → Shard {new}")
print("Directory update is instant — but you still need to physically move the data!")

## Step 5: Strategy 3 — Shard Splitting

When a shard gets too large, you can **split it** into two shards. This is how MongoDB's balancer works:

1. Pick the overloaded shard
2. Split its data roughly in half
3. Move one half to a new shard
4. Update routing to reflect the split

In [ ]:
def simulate_shard_split(shard_assignments):
    """
    Simulate splitting the most overloaded shard.
    We can't create a 4th Postgres instance at runtime,
    so we'll simulate the split logic using the directory.
    """
    # Find the most overloaded shard
    shard_counts = {}
    for uid, sid in shard_assignments.items():
        shard_counts[sid] = shard_counts.get(sid, 0) + 1
    
    overloaded = max(shard_counts, key=shard_counts.get)
    users_on_shard = [uid for uid, sid in shard_assignments.items() if sid == overloaded]
    users_on_shard.sort()
    
    # Split in half
    midpoint = len(users_on_shard) // 2
    stay = users_on_shard[:midpoint]
    move = users_on_shard[midpoint:]
    
    # In reality, the moved half goes to a NEW shard.
    # For simulation, we'll track it as shard 3 (virtual).
    new_shard_id = max(shard_counts.keys()) + 1
    
    new_assignments = dict(shard_assignments)
    for uid in move:
        new_assignments[uid] = new_shard_id
    
    return overloaded, new_shard_id, len(stay), len(move), new_assignments

src, new_sid, stayed, moved_count, new_assigns = simulate_shard_split(shard_assignments)

print("🔪 Shard Split Simulation")
print("═" * 50)
print(f"Split Shard {src} into Shard {src} + Shard {new_sid}")
print(f"  Users staying on Shard {src}: {stayed}")
print(f"  Users moved to Shard {new_sid}:  {moved_count}")
print()

# Show new distribution
new_counts = {}
for uid, sid in new_assigns.items():
    new_counts[sid] = new_counts.get(sid, 0) + 1

print("New distribution after split:")
print("─" * 40)
for sid in sorted(new_counts):
    count = new_counts[sid]
    bar = '█' * (count // 3)
    label = ' (NEW)' if sid == new_sid else ''
    print(f"  Shard {sid}: {count:3d} users {bar}{label}")

## Step 6: Monitoring — Keeping Shards Healthy

Rebalancing isn't a one-time thing. You need continuous monitoring to detect when shards become unbalanced again. Let's build a simple monitor.

In [ ]:
def shard_health_check(threshold=1.5):
    """
    Check shard balance and report any problems.
    A shard is 'hot' if it has more than threshold * average rows.
    """
    stats = get_shard_stats()
    total = sum(s['users'] for s in stats.values())
    avg = total / len(stats) if stats else 0
    
    print("🏥 Shard Health Check")
    print("═" * 60)
    
    alerts = []
    for sid, s in stats.items():
        ratio = s['users'] / avg if avg > 0 else 0
        
        if ratio > threshold:
            status = '🔴 HOT'
            alerts.append(f"Shard {sid} is {ratio:.1f}x the average — consider splitting")
        elif ratio < 1/threshold:
            status = '🟡 COLD'
            alerts.append(f"Shard {sid} is underutilized — consider merging")
        else:
            status = '🟢 OK'
        
        bar_len = int(s['users'] / max(1, avg) * 20)
        bar = '█' * bar_len
        print(f"  Shard {sid}: {s['users']:4d} users | {s['orders']:4d} orders | {status} {bar}")
    
    print(f"\n  Average: {avg:.0f} users/shard")
    
    if alerts:
        print("\n⚠️  Alerts:")
        for alert in alerts:
            print(f"  → {alert}")
    else:
        print("\n✅ All shards are balanced!")

shard_health_check()

## Step 7: Comparison of Rebalancing Strategies

Let's summarize when to use each approach.

In [ ]:
print("📋 Rebalancing Strategy Comparison")
print("═" * 75)
print()
print("┌─────────────────────┬──────────────────────┬──────────────────────────────┐")
print("│ Strategy            │ Best For             │ Tradeoff                     │")
print("├─────────────────────┼──────────────────────┼──────────────────────────────┤")
print("│ Online Migration    │ Moving specific keys │ Slow for large datasets      │")
print("│ Shard Splitting     │ Growing shards       │ Need to provision new shards │")
print("│ Directory Routing   │ Maximum flexibility  │ Extra lookup per request      │")
print("│ Consistent Hashing  │ Adding/removing nodes│ Requires virtual nodes       │")
print("└─────────────────────┴──────────────────────┴──────────────────────────────┘")
print()
print("In practice, most systems combine these:")
print("  1. Use consistent hashing for initial distribution")
print("  2. Monitor for hot spots")
print("  3. Use online migration to rebalance when needed")
print("  4. Split shards when they outgrow their hardware")
print()
print("Modern databases handle much of this automatically:")
print("  • MongoDB's balancer splits and migrates chunks")
print("  • DynamoDB splits/merges partitions transparently")
print("  • Vitess supports operator-driven online resharding")
print("  • Cassandra uses virtual nodes for automatic balance")

## Step 8: The Real-World Migration Pattern — Dual Writes 🎯

The migration we did earlier worked because **no one was writing to the system during the migration**. In production that's never true — users are placing orders, updating profiles, etc., 24/7.

The standard zero-downtime migration pattern has **four phases**:

| Phase | What happens | Reads from | Writes to |
|-------|--------------|------------|-----------|
| 1. **Dual-write** | Write to both old & new shard | Old | Old **and** New |
| 2. **Backfill**   | Copy old historical data to new shard | Old | Old **and** New |
| 3. **Verify**     | Compare row counts / checksums across both | Old | Old **and** New |
| 4. **Cutover**    | Flip reads to new shard, stop writing to old | **New** | New only |

If anything goes wrong during verification, you roll back by flipping reads back to the old shard. No data lost.


In [ ]:
class DualWriteMigrator:
    """
    Simulates a zero-downtime migration using the dual-write pattern.

    Phase 1 (dual-write): Every write goes to BOTH the old and new shard.
    Phase 2 (backfill):   Copy pre-existing rows from old → new.
    Phase 3 (verify):     Confirm both shards have identical data.
    Phase 4 (cutover):    Flip reads to the new shard.
    """

    def __init__(self, old_shard, new_shard):
        self.old_shard = old_shard
        self.new_shard = new_shard
        self.phase = "dual_write"  # dual_write → backfill → verify → cutover

    def write_user(self, uid, username, email, country):
        """During migration, ALL writes go to both shards."""
        for sid in [self.old_shard, self.new_shard]:
            conn = get_connection(sid)
            conn.autocommit = True
            cur = conn.cursor()
            cur.execute(
                "INSERT INTO users (id, username, email, country) VALUES (%s,%s,%s,%s) "
                "ON CONFLICT (id) DO NOTHING",
                (uid, username, email, country),
            )
            cur.close()
            conn.close()

    def read_user(self, uid):
        """Reads go to 'old' until cutover, then to 'new'."""
        target = self.new_shard if self.phase == "cutover" else self.old_shard
        conn = get_connection(target)
        cur = conn.cursor()
        cur.execute("SELECT id, username FROM users WHERE id = %s", (uid,))
        row = cur.fetchone()
        cur.close()
        conn.close()
        return target, row

    def backfill(self, user_ids):
        """Copy existing rows from old → new shard."""
        self.phase = "backfill"
        src = get_connection(self.old_shard); src.autocommit = True
        dst = get_connection(self.new_shard); dst.autocommit = True
        sc, dc = src.cursor(), dst.cursor()
        for uid in user_ids:
            sc.execute("SELECT id, username, email, country FROM users WHERE id=%s", (uid,))
            r = sc.fetchone()
            if r:
                dc.execute(
                    "INSERT INTO users (id, username, email, country) VALUES (%s,%s,%s,%s) "
                    "ON CONFLICT (id) DO NOTHING", r,
                )
        sc.close(); dc.close(); src.close(); dst.close()

    def verify(self, user_ids):
        """Confirm the two shards agree on every row."""
        self.phase = "verify"
        mismatches = 0
        for sid in [self.old_shard, self.new_shard]:
            pass
        src = get_connection(self.old_shard); dst = get_connection(self.new_shard)
        sc, dc = src.cursor(), dst.cursor()
        for uid in user_ids:
            sc.execute("SELECT username, email FROM users WHERE id=%s", (uid,))
            dc.execute("SELECT username, email FROM users WHERE id=%s", (uid,))
            if sc.fetchone() != dc.fetchone():
                mismatches += 1
        sc.close(); dc.close(); src.close(); dst.close()
        return mismatches

    def cutover(self):
        """Flip reads to the new shard. Old shard can now be retired."""
        self.phase = "cutover"


# Demo: migrate a handful of users from shard 0 to shard 2 using dual-write
mig = DualWriteMigrator(old_shard=0, new_shard=2)

# Phase 1: new writes go to both shards from this moment on
print("Phase 1 — Dual-writing new signups to both shards")
for uid in range(901, 906):
    mig.write_user(uid, f"user_{uid}", f"u{uid}@ex.com", "US")
print(f"  Wrote users 901–905 to BOTH shards {mig.old_shard} and {mig.new_shard}")

# Phase 2: backfill historical data (existing users on shard 0)
print("\nPhase 2 — Backfilling historical rows from old → new")
existing_on_old = [u for u, s in shard_assignments.items() if s == 0][:10]
mig.backfill(existing_on_old)
print(f"  Backfilled {len(existing_on_old)} historical users")

# Phase 3: verify
print("\nPhase 3 — Verifying both shards match")
bad = mig.verify(existing_on_old + list(range(901, 906)))
print(f"  Mismatches: {bad}  {'✅ safe to cutover' if bad == 0 else '❌ investigate!'}")

# Phase 4: cutover
print("\nPhase 4 — Cutover: reads now served from the NEW shard")
mig.cutover()
target, row = mig.read_user(901)
print(f"  Read of user 901 routed to Shard {target} → {row}")


### Why this matters

Every time a company "resharded" a live production system, it followed some version of this pattern:

- **Shopify** migrated to "pods" using dual-writes then cutover
- **Stripe**'s ledger resharding ran in dual-write mode for weeks
- **Slack**'s shard splits use a similar backfill + verify + flip flow
- **Vitess** (the sharding layer for YouTube's MySQL) automates these phases

The key insight: you can **always roll back** until cutover. After cutover, the old shard becomes read-only and is eventually decommissioned.


## 🎯 Key Takeaways

1. **Monitor continuously** — detect imbalance before it becomes a crisis
2. **Online migration** is the simplest approach: copy data, update routing, delete old
3. **Directory-based routing** gives maximum flexibility but adds a lookup dependency
4. **Shard splitting** handles organic growth by dividing a large shard in two
5. **Design to minimize rebalancing** — good shard keys and consistent hashing prevent most issues
6. In interviews: "We'll use consistent hashing for distribution and plan for operator-driven resharding when needed."

## 🎓 Series Summary

| Notebook | Strategy | Use When |
|----------|----------|----------|
| 1. Hash-Based | `hash(key) % N` | Default choice, even distribution |
| 2. Range-Based | Value ranges | Range scans, multi-tenant systems |
| 3. Consistent Hashing | Hash ring | Need to add/remove shards safely |
| 4. Rebalancing | Migration & splitting | Shards become unbalanced over time |